# Reproducing Bouchard et al. (2013) from dandiset 000019

Paper. Bouchard KE, Mesgarani N, Johnson K, Chang EF (2013). Functional organization of
human sensorimotor cortex for speech articulation. *Nature* 495, 327-332.
[doi:10.1038/nature11911](https://doi.org/10.1038/nature11911). The claims file
(`claims/000019/10.1038-nature11911.yaml`) was written from the NIH author manuscript in PubMed
Central (PMC3606666) with its supplement, and checked by a second reader.

Dandiset. [DANDI:000019](https://dandiarchive.org/dandiset/000019/0.220126.2148),
"Human ECoG speaking consonant-vowel syllables", version 0.220126.2148. The paper cites no
version of the dandiset (it predates it). This is the only published version; the draft has the
same 31 assets and the same size, so the published version is pinned.

What the dandiset holds. Raw ECoG (256 channels, 3051.76 Hz) of 31 recording blocks from four
subjects (EC2, EC9, GP31, GP33), with a trial table per block (syllable, start, C-V transition
and stop times), a `bad` flag and an anatomical label per electrode, 3-D electrode coordinates for
two subjects, and a rest period per block. The microphone signal was removed (dandiset
description), and no high gamma is stored, so high gamma is computed here from the raw voltage.
The paper's subjects are EC2, EC33 and EC31; EC2, GP33 and GP31 are taken to be them (see the data
check). EC9 is not in the paper and is not analysed.

Readings. Where the claims file leaves a method detail open it names readings, and every
reading is run; a claim's outcome is the one all its method readings give. Two readings are added
here as questions to the reader (`questions/000019.yaml`): how the state-space input is z-scored
(`row`: each electrode's concatenated series; `time`: each electrode at each time across
syllables), and the dataset's rest period as a baseline (shown as a variant only).

| Claim | Tested here |
|---|---|
| data-composition | tested |
| active-electrodes | tested |
| fig1c-e-articulator-electrodes | tested |
| fig1f-h-phonetic-contrasts | tested |
| fig2a-lips-dorsal | blocked: metadata_insufficient (no positions relative to the central sulcus and Sylvian fissure) |
| fig2b-somatotopic-sequence | blocked: metadata_insufficient |
| suppfig3b-spatial-clustering | blocked: metadata_insufficient |
| suppfig3d-cross-subject | blocked: metadata_insufficient |
| suppfig3f-dv-sequence | blocked: metadata_insufficient |
| suppfig5-intermediate-organization | blocked: metadata_insufficient |
| suppfig6-mixed-tuning | tested |
| fig3d-timing-sequence | tested |
| suppfig7-no-latency-difference | blocked: metadata_insufficient |
| pca-nine-components | tested |
| fig4a-consonant-clusters | tested |
| suppfig11-co-clustering | tested |
| fig4c-d-hierarchy | tested |
| fig4ef-feature-correlations | tested |
| fig5c-separability-timing | tested |
| fig5d-separability-coupling | tested |
| suppfig13-consonant-vowel-distance | tested |
| suppfig9-pc-articulator-weights | tested |

In [ ]:
import itertools
import time
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import remfile
from scipy import stats
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import silhouette_score

import bouchard as B

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()

In [ ]:
# Metadata of every file (EC9 included), and a fingerprint of its first samples, read by streaming.
ASSETS = B.assets()
meta_rows = []
for a in ASSETS:
    with h5py.File(remfile.File(a["url"]), "r") as h5:
        m = B.read_metadata(h5)
        m["fingerprint"] = h5["acquisition/ElectricalSeries/data"][:2000, :4].copy()
    m["path"], m["size"] = a["path"], a["size"]
    meta_rows.append(m)
print(f"read metadata of {len(meta_rows)} files of dandiset {B.DANDISET_ID} version {B.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

In [ ]:
# High gamma of the three subjects of the paper: every block is streamed into memory and processed.
blocks = {}
for s in B.SUBJECTS:
    blocks[s] = B.process_blocks([a for a in ASSETS if a["path"].startswith(f"sub-{s}/")],
                                 log=lambda msg: None)
S = {s: B.load_subject(blocks[s]) for s in B.SUBJECTS}
print(f"processed {sum(len(v) for v in blocks.values())} blocks in {time.time() - started:.0f} s")

## Data check

In [ ]:
md = B.dandiset_metadata()
print("Dandiset:", md["name"], "| version", B.DANDISET_VERSION, "| DOI", md.get("doi"))
print("Related resource:", [(r.get("relation"), r.get("identifier"), r.get("url")) for r in md.get("relatedResource", [])])
print("Assets summary: subjects", md["assetsSummary"].get("numberOfSubjects"), "| files", md["assetsSummary"].get("numberOfFiles"))
files = pd.DataFrame([{
    "file": m["path"].split("/")[1], "subject": m["subject"], "rate_hz": m["rate"],
    "minutes": m["n_samples"] / m["rate"] / 60, "channels": m["n_channels"], "unit": m["unit"],
    "conversion": m["conversion"], "trials": len(m["condition"]), "speak": int(m["speak"].sum()),
    "bad": int(m["bad"].sum()), "rest_s": sum(b - a for a, b in m["rest"]),
    "invalid_intervals": len(m["invalid"]), "has_xyz": bool(np.isfinite(m["x"]).any()),
    "mean_x": np.nanmean(m["x"]) if np.isfinite(m["x"]).any() else np.nan,
    "electrode_ids": f"{m['electrode_ids'][0]}-{m['electrode_ids'][-1]}"} for m in meta_rows])
print(files.round(2).to_string(index=False))
print("\nFiles and trials per subject:")
print(files.groupby("subject").agg(files=("file", "size"), trials=("trials", "sum"), minutes=("minutes", "sum")).round(1).to_string())
print("\nSampling rates found:", sorted(files.rate_hz.unique()), "(Methods: 3052 Hz)")
print("Trials not marked speak=True, per subject:", (files.trials - files.speak).groupby(files.subject).sum().to_dict())
print("Microphone or other acquisition series in any file: none (only acquisition/ElectricalSeries is present)")

In [ ]:
# Duplicates: blocks with the same length, the same trial table, or the same first samples.
pairs = []
for i, j in itertools.combinations(range(len(meta_rows)), 2):
    a, b = meta_rows[i], meta_rows[j]
    same_n = a["n_samples"] == b["n_samples"]
    same_trials = len(a["cv"]) == len(b["cv"]) and np.allclose(a["cv"], b["cv"])
    same_data = np.array_equal(a["fingerprint"], b["fingerprint"])
    if same_n or same_trials or same_data:
        pairs.append((a["path"], b["path"], same_n, same_trials, same_data))
print("Pairs sharing sample count, trial times or first 2000 samples of 4 channels:")
for p in pairs:
    print("  ", p)

Only one pair of files shares a property (two EC9 blocks with the same number of samples); their
trial times and first samples differ, so no recording was deposited twice.

In [ ]:
# Syllables per subject, from the trial labels.
label_rows = []
for m in meta_rows:
    for c in m["condition"]:
        label_rows.append({"subject": m["subject"], "label": c, "cv": B.parse_label(c)})
labels = pd.DataFrame(label_rows)
for s, g in labels.groupby("subject"):
    unparsed = g[g.cv.isna()].label.value_counts().to_dict()
    cv = g.dropna().cv
    cons = sorted({c for c, v in cv}, key=B.CONSONANTS.index)
    counts = cv.value_counts()
    print(f"{s}: {len(g)} trials, {counts.size} distinct CVs, {len(cons)} consonants; renditions per CV "
          f"{counts.min()} to {counts.max()}; labels that are not CVs of the task: {unparsed}")
    missing = [c + v for c in B.CONSONANTS for v in "aiu" if (c, v) not in set(cv)]
    print(f"    CVs missing: {missing}; CVs with fewer than 15 renditions: "
          f"{ {''.join(k): int(n) for k, n in counts.items() if n < 15} }; more than 100: "
          f"{ {''.join(k): int(n) for k, n in counts.items() if n > 100} }")
print("Labels merged as misspellings:", labels[labels.label.isin(list(B.SPELLING))].groupby(["subject", "label"]).size().to_dict())
print("Spellings merged: 'she' -> 'shee', 'ha' -> 'haa'. Paper: 19, 19 and 18 consonants, 168 CVs (57 + 57 + 54), 15 to 100 renditions each.")

In [ ]:
# Electrodes: labels, bad flags, coordinates.
for s in ["EC2", "EC9", "GP31", "GP33"]:
    ms = [m for m in meta_rows if m["subject"] == s]
    same_loc = all(np.array_equal(m["location"], ms[0]["location"]) for m in ms)
    loc = pd.Series(ms[0]["location"]).replace("", "(none)").value_counts().to_dict()
    badsets = sorted({tuple(np.where(m["bad"])[0]) for m in ms}, key=len)
    print(f"{s}: labels identical in all blocks {same_loc}; labels {loc}")
    print(f"    bad-channel sets over blocks: {[list(b) for b in badsets]}")
    x = ms[0]["x"]
    if np.isfinite(x).any():
        print(f"    x {np.nanmin(x):.1f} to {np.nanmax(x):.1f}, y {np.nanmin(ms[0]['y']):.1f} to {np.nanmax(ms[0]['y']):.1f}, "
              f"z {np.nanmin(ms[0]['z']):.1f} to {np.nanmax(ms[0]['z']):.1f} (no unit or frame given); "
              f"neighbour spacing median {np.median(np.linalg.norm(np.diff(np.c_[x, ms[0]['y'], ms[0]['z']], axis=0), axis=1)):.2f}")
    else:
        print("    no coordinates (x, y, z are NaN)")
for s in B.SUBJECTS:
    print(f"{s}: vSMC (precentral or postcentral, good in every block) {len(S[s]['vsmc'])} electrodes; "
          f"electrodes bad in any block {int((~S[s]['good']).sum())}")
print("vSMC electrodes over the three subjects:", sum(len(S[s]["vsmc"]) for s in B.SUBJECTS), "(Supplementary Figure 9: n = 266)")

The archive's summary counts four subjects. EC2 matches the paper's EC2 by name. GP31 and GP33
are not names the paper uses; the paper's other two subjects are EC31 (left hemisphere) and
EC33 (right). GP31's coordinates have negative x in every electrode, which fits a left
hemisphere if x is the usual left-right axis; GP33 has no coordinates, so its hemisphere cannot be
checked. That GP31 is EC31 and GP33 is EC33 is an assumption the data are consistent with, not
shown by them. EC9 has 24 consonant labels (including /ch/, /dh/, /j/, /sk/), labels on only 79 of
256 electrodes and no coordinates; it is not one of the paper's subjects and is left out.

In [ ]:
# Trials kept, trials in invalid intervals, rest periods, and the high-gamma statistics.
rows = []
for s in B.SUBJECTS:
    for b in blocks[s]:
        cv = b["cv"]
        inv = sum(((cv[b["kept"]] + 0.75 > a0) & (cv[b["kept"]] - 0.5 < a1)).sum() for a0, a1 in b["invalid"])
        g = ~b["bad"]
        rows.append({"subject": s, "block": b["session"], "trials": len(cv), "kept": len(b["kept"]),
                     "kept_in_invalid": int(inv), "rest_s": round(b["rest_seconds"], 1),
                     "hg_whole_mean_uV": float(np.nanmedian(b["hg_stats"]["whole"][0][g]) * 1e6),
                     "hg_whole_sd_over_mean": float(np.nanmedian(b["hg_stats"]["whole"][1][g] / b["hg_stats"]["whole"][0][g])),
                     "pre_mean_over_whole": float(np.nanmedian(b["hg_stats"]["pre"][0][g] / b["hg_stats"]["whole"][0][g])),
                     "rest_mean_over_whole": float(np.nanmedian(b["hg_stats"]["rest"][0][g] / b["hg_stats"]["whole"][0][g])),
                     "rest_sd_over_whole": float(np.nanmedian(b["hg_stats"]["rest"][1][g] / b["hg_stats"]["whole"][1][g])),
                     "env_rate": round(b["env_rate"], 3)})
blk = pd.DataFrame(rows)
print(blk.round(3).to_string(index=False))
print("\nTrials dropped (epoch -1000 to +750 ms outside the recording):", int((blk.trials - blk.kept).sum()))
print("Kept trials whose epoch overlaps an 'ECoG artifact' invalid interval (not excluded; the claims file does not say to):",
      int(blk.kept_in_invalid.sum()), "of", int(blk.kept.sum()))
sd_raw = {s: float(np.median([np.std(m["fingerprint"], axis=0).mean() for m in meta_rows if m["subject"] == s])) for s in B.SUBJECTS}
print("Raw series: unit", meta_rows[0]["unit"], "with conversion", meta_rows[0]["conversion"],
      "| median SD of the stored values over the first 2000 samples of 4 channels:", {k: f"{v:.1e}" for k, v in sd_raw.items()})
print("hg_whole_mean_uV above is the stored high-gamma amplitude x 1e6, without the conversion factor.")
for s in B.SUBJECTS:
    for base in B.BASELINES:
        H = S[s]["H"][base][:, S[s]["vsmc"]]
        print(f"{s} {base:5s}: per-CV mean z over vSMC electrodes, range {np.nanmin(H):.2f} to {np.nanmax(H):.2f}; "
              f"mean over -500 to -400 ms {np.nanmean(H[:, :, :21]):.2f}")

Units: the raw series says volts with a conversion factor of 0.001, so by NWB the stored values
times 0.001 are volts. The stored values have an SD of a few 1e-5 (printed above): read as volts
this is tens of microvolts, a usual ECoG amplitude, while with the factor it would be tens of
nanovolts. The data are consistent with values already in volts and a wrong conversion factor;
this does not affect the analysis, which uses z-scores only.
Sampling rate 3051.7578 Hz in all files, the TDT rate the Methods round to 3052 Hz. The `pre`
baseline (-1000 to -500 ms before each C-V transition) has a median mean within a few percent of
the whole block's. The rest periods are short (3.3 to 78 s per block) and their statistics differ
from the whole block's in both directions by block (median ratios above); z-scores against them
reach higher maxima (ranges above). That variant is shown where it matters.

In [ ]:
# Clock check: the trial times and the ECoG share a clock if the high gamma tells syllables apart
# at the times the paper says, and not before speech. Leave-one-out nearest-centroid classification
# of the vowel (/a/, /i/, /u/) and of the major oral articulator of the consonant (labial, coronal,
# dorsal; reading (a) of the labels) from the vSMC high-gamma pattern of each CV, at each time.
def loo_accuracy(X, y):
    ok = 0
    for i in range(len(y)):
        mask = np.arange(len(y)) != i
        cents = {c: X[mask & (y == c)].mean(0) for c in np.unique(y[mask])}
        pred = min(cents, key=lambda c: np.linalg.norm(X[i] - cents[c]))
        ok += pred == y[i]
    return ok / len(y)

clock = []
for s in B.SUBJECTS:
    H = S[s]["H"]["pre"][:, S[s]["vsmc"]]
    vow = np.array([v for c, v in S[s]["cvs"]])
    maj = np.array([B.MAJOR["a"].get(c, "") for c, v in S[s]["cvs"]])
    lab = maj != ""
    for t in range(0, len(B.EPOCH_MS), 5):
        clock.append({"subject": s, "t_ms": B.EPOCH_MS[t], "vowel": loo_accuracy(H[:, :, t], vow),
                      "consonant": loo_accuracy(H[lab][:, :, t], maj[lab])})
clock = pd.DataFrame(clock)
summary = clock.groupby("t_ms")[["vowel", "consonant"]].mean()
print("Mean accuracy over subjects (chance 1/3 for vowels; 0.5 to 0.53 for consonants, the largest class share):")
print(summary.iloc[::4].round(2).T.to_string())
for col in ["vowel", "consonant"]:
    print(f"{col}: peak {summary[col].max():.2f} at {summary[col].idxmax()} ms; mean before -400 ms {summary[col][summary.index < -400].mean():.2f}")
fig, ax = plt.subplots(figsize=(6, 3))
summary.plot(ax=ax)
ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("time from C-V transition (ms)"); ax.set_ylabel("LOO accuracy")
plt.show()

The consonant's articulator is decoded best before the C-V transition (0.89 at -75 ms) and the
vowel after it (0.83 at 250 ms), and both are near chance at the start of the epoch (vowel 0.39
against 1/3; consonant 0.59 against a largest-class share of about 0.5). This is
consistent with the trial times and the ECoG sharing a clock in every subject, with the
annotations marking the C-V transition; it does not show the alignment to the millisecond.

## Shared computations

The partial correlations and the GLM (Methods Eqs. 2 to 4) are computed once per baseline, and
the state space (Methods Eq. 5) once per baseline and z-scoring reading. The articulator windows
T_A come from the timing analysis under each of its eight readings (claim fig3d-timing-sequence:
threshold set per subject or pooled, over all six articulators or per articulator, and onset at the
threshold or at half the peak).

In [ ]:
BASES = ("pre", "whole")
ZMODES = ("row", "time")
ART6 = B.ARTICULATORS + B.VOWEL_ARTICULATORS
TIMING_READINGS = list(itertools.product(("per subject", "pooled"), ("all six", "per articulator"), ("threshold", "half peak")))
CVSETS = ("all", "15+")    # every CV, or only CVs with at least 15 renditions (a question to the reader)
CVIDX = {(s, cs): np.where(S[s]["counts"] >= (15 if cs == "15+" else 0))[0] for s in B.SUBJECTS for cs in CVSETS}
CVS = {k: [S[k[0]]["cvs"][i] for i in v] for k, v in CVIDX.items()}
AM = {k: B.articulator_matrix(v) for k, v in CVS.items()}
HV = {(s, base, cs): S[s]["H"][base][CVIDX[s, cs]][:, S[s]["vsmc"]]
      for s in B.SUBJECTS for base in B.BASELINES for cs in CVSETS}
RHO = {(s, base, cs): B.partial_correlation(HV[s, base, cs], AM[s, cs]) for s in B.SUBJECTS for base in BASES for cs in CVSETS}
GLM = {(s, base, cs): B.glm(HV[s, base, cs], AM[s, cs][:, :4]) for s in B.SUBJECTS for base in BASES for cs in CVSETS}


def timing_rows(base, cs, reading):
    subj_mode, art_mode, onset_mode = reading
    rows = []
    for s in B.SUBJECTS:
        pool = np.concatenate([RHO[x, base, cs] for x in B.SUBJECTS]) if subj_mode == "pooled" else RHO[s, base, cs]
        for a in range(6):
            vals = pool if art_mode == "all six" else pool[:, a]
            thr = vals.mean() + 2.5 * vals.std()
            for e in range(RHO[s, base, cs].shape[0]):
                r = RHO[s, base, cs][e, a]
                if r.max() > thr:
                    on, pk, off = B.timing(r, thr if onset_mode == "threshold" else r.max() / 2)
                    rows.append({"subject": s, "articulator": ART6[a], "electrode": e, "onset": on,
                                 "peak": pk, "offset": off, "threshold": thr})
    return pd.DataFrame(rows)


TIMING = {(base, cs, r): timing_rows(base, cs, r) for base in BASES for cs in CVSETS for r in TIMING_READINGS}


def articulator_windows(base, cs, reading):
    """T_A per subject: mean onset to mean offset of the subject's electrodes for A (None if none)."""
    t = TIMING[base, cs, reading]
    out = {}
    for s in B.SUBJECTS:
        for a in B.ARTICULATORS:
            g = t[(t.subject == s) & (t.articulator == a)]
            out[s, a] = (g.onset.mean(), g.offset.mean()) if len(g) else None
    return out


def weights(base, cs, reading, pmode):
    """Per subject: W (vSMC electrode, 4), NaN for an articulator without a window, and the
    electrodes kept by the GLM rule (mean p over T_A below 0.05 for any articulator)."""
    T = articulator_windows(base, cs, reading)
    out = {}
    for s in B.SUBJECTS:
        beta, pF, pt, r2 = GLM[s, base, cs]
        keep = np.zeros(beta.shape[0], bool)
        W = np.full((beta.shape[0], 4), np.nan)
        for a, name in enumerate(B.ARTICULATORS):
            if T[s, name] is None:
                continue
            idx = (B.EPOCH_MS >= T[s, name][0]) & (B.EPOCH_MS <= T[s, name][1])
            p = (pF[:, idx] if pmode == "F-test" else pt[:, a, idx]).mean(1)
            keep |= p < 0.05
            W[:, a] = beta[:, a, idx].mean(1)
        out[s] = {"W": W, "keep": keep, "windows": {a: T[s, a] for a in B.ARTICULATORS}}
    return out


W_READINGS = [(base, cs, r, p) for base in BASES for cs in CVSETS for r in TIMING_READINGS for p in ("F-test", "t-test")]
WEIGHTS = {k: weights(*k) for k in W_READINGS}
STATE = {(s, base, z, cs, w): B.state_space(HV[s, base, cs], z, w)
         for s in B.SUBJECTS for base in BASES for z in ZMODES for cs in CVSETS
         for w in [(-500, 750), (-375, 620)]}
SS_READINGS = list(itertools.product(BASES, ZMODES, CVSETS))
EP = (-500, 750)
print(f"shared computations done at {time.time() - started:.0f} s")

## data-composition

> Three subjects with high-density ECoG (two left, one right hemisphere) read aloud CV syllables
> made of 18 or 19 consonants followed by /a/, /i/ or /u/, each CV produced 15 to 100 times, with
> the microphone recorded with the ECoG at 3052 Hz.

In [ ]:
comp = []
for s in B.SUBJECTS:
    n = S[s]["counts"]
    comp.append({"subject": s, "CVs": len(S[s]["cvs"]), "consonants": len({c for c, v in S[s]["cvs"]}),
                 "vowels": len({v for c, v in S[s]["cvs"]}), "min_renditions": int(n.min()), "max_renditions": int(n.max()),
                 "CVs_below_15": int((n < 15).sum()), "CVs_above_100": int((n > 100).sum()),
                 "consonants_with_all_3_vowels_and_15+": len([c for c in B.CONSONANTS if all((c, v) in S[s]["cvs"] and n[S[s]["cvs"].index((c, v))] >= 15 for v in "aiu")]),
                 "hemisphere_from_x": ("left" if np.nanmean(S[s]["x"]) < 0 else "right") if np.isfinite(S[s]["x"]).any() else "unknown"})
comp = pd.DataFrame(comp)
print(comp.to_string(index=False))
print("Subjects in the dandiset:", sorted(files.subject.unique()), "| of the paper's (EC2, EC31, EC33), by assumed correspondence: 3")
print("Distinct CVs over the three subjects:", comp.CVs.sum(), "(paper 168); with CVs of fewer than 15 renditions left out:",
      int((comp.CVs - comp.CVs_below_15).sum()))
print("ECoG sampling rate:", sorted(files.rate_hz.unique()), "Hz (3052 +- 1)")
print("Audio in the files: none (removed for HIPAA, dandiset description)")

**Result: partially reproduced.** The dandiset holds the three subjects (taking GP31 and GP33 to
be EC31 and EC33) and a fourth one, EC9, that is not in the paper. The ECoG rate is 3051.76 Hz.
The counts differ from the paper's: 57, 55 and 57 CVs (169 in all, against 168), each subject with
19 consonants, and renditions of 2 to 107 per CV (EC2's /thi/ 10, GP31's /thi/ 2, GP31 up to 107).
Leaving out CVs with fewer than 15 renditions gives 56 + 54 + 57 = 167, still not the paper's
split. The audio is not in the dandiset, and GP33's hemisphere cannot be checked.

## active-electrodes

> About 30 active vSMC electrodes were found per subject during syllable articulation, an
> electrode being active when its high-gamma z-score changed by more than 2 for any syllable.

In [ ]:
def overall(outcomes):
    """The outcome all method readings give, or methods_underspecified when they differ."""
    u = sorted(set(outcomes))
    return u[0] if len(u) == 1 else "differ by reading: " + ", ".join(f"{o} {list(outcomes).count(o)}" for o in u)


act = []
for base in B.BASELINES:
    for cs in CVSETS:
        for s in B.SUBJECTS:
            H = HV[s, base, cs]
            act.append({"baseline": base, "CVs": cs, "subject": s, "vSMC": H.shape[1],
                        "(a) z > 2": int((np.nanmax(H, axis=(0, 2)) > 2).sum()),
                        "(b) |z| > 2": int((np.nanmax(np.abs(H), axis=(0, 2)) > 2).sum())})
act = pd.DataFrame(act)
print(act.to_string(index=False))
am = act.groupby(["baseline", "CVs"])[["(a) z > 2", "(b) |z| > 2"]].mean().round(2)
am["within 27-33 (a or b)"] = am.apply(lambda r: 27 <= r["(a) z > 2"] <= 33 or 27 <= r["(b) |z| > 2"] <= 33, axis=1)
print("\nMean over subjects (criterion 27 to 33; rest is a variant, not a reading):")
print(am.to_string())
for thr in (1.5, 2, 2.5, 3):
    print(f"threshold {thr} (all CVs): mean active per subject, pre {np.mean([(np.nanmax(HV[s, 'pre', 'all'], axis=(0, 2)) > thr).sum() for s in B.SUBJECTS]):.1f}, "
          f"whole {np.mean([(np.nanmax(HV[s, 'whole', 'all'], axis=(0, 2)) > thr).sum() for s in B.SUBJECTS]):.1f}")
res = am.loc[list(BASES)]
print("Outcome over readings:", overall(["reproduced" if all(res.xs(cs, level=1)["within 27-33 (a or b)"]) else
                                         ("partially" if any(res.xs(cs, level=1)["within 27-33 (a or b)"]) else "not reproduced")
                                         for cs in CVSETS]))

**Result: not reproduced.** With every CV, the mean count is 24.3 per subject with the `pre`
baseline (EC2 15, GP31 49, GP33 9) and 21.3 with the whole block; with only the CVs of 15 or more
renditions it is lower still (table). Both are below 27 under either reading of the change, as no
electrode passes on a decrease alone. With the dataset's rest period as baseline (a variant) the
mean is 34.0 with every CV. The subjects differ widely, and the count moves with the threshold.

## fig1c-e-articulator-electrodes

> In one subject, dorsal electrodes (e124, e108) were active for /b/ (lips), mid vSMC electrodes
> (e129, e133, e105) for /d/ (front tongue), and a more ventral electrode (e104) was most active
> for /g/ (back tongue).

In [ ]:
NAMED = {"b": [124, 108], "d": [129, 133, 105], "g": [104]}
f1 = []
for s in B.SUBJECTS:
    for numbering, offset in (("(a) from 1", 1), ("(b) from 0", 0)):
        idx = {k: [n - offset for n in v] for k, v in NAMED.items()}
        all_idx = sum(idx.values(), [])
        on_vsmc = [i in S[s]["vsmc"] for i in all_idx]
        for base in BASES:
            for measure in ("(i) at -25 ms", "(ii) peak"):
                H = S[s]["H"][base]
                resp = {}
                for c in "bdg":
                    k = S[s]["cvs"].index((c, "a"))
                    resp[c] = H[k, :, B.ms_index(-25)] if measure.startswith("(i)") else np.nanmax(H[k], axis=1)
                prefers = {i: max("bdg", key=lambda c: resp[c][i]) for i in all_idx}
                ok = [prefers[i] == c for c, ii in idx.items() for i in ii]
                f1.append({"subject": s, "numbering": numbering, "baseline": base, "measure": measure,
                           "named on vSMC": f"{sum(on_vsmc)} of 6",
                           "labels": ",".join(S[s]["location"][i][:4] + ("" if S[s]["good"][i] else "(bad)") for i in all_idx),
                           "preferences (b b d d d g)": " ".join(prefers[i] for i in all_idx),
                           "prefer their syllable": f"{sum(ok)} of 6"})
f1 = pd.DataFrame(f1)
print(f1.to_string(index=False))
print("\nSubject and numbering pairs that put all six named electrodes on vSMC:",
      f1[f1["named on vSMC"] == "6 of 6"][["subject", "numbering"]].drop_duplicates().values.tolist())

**Result: blocked (metadata_insufficient).** No subject, counted from 1 or from 0, has all six
named electrodes on vSMC (at most 4 of 6, in EC2 and GP31; some named electrodes are labelled
middle or superior temporal, or pars opercularis). The criterion blocks the claim in that case:
the dataset does not say which subject Fig. 1 shows or how the paper numbered electrodes. For
information, in EC2 counted from 1, five of the six named electrodes prefer their syllable under
both measures and both baselines.

## fig1f-h-phonetic-contrasts

> Fricatives with different tongue-tip constriction locations engaged different electrodes;
> front-tongue consonants of different manner engaged overlapping electrodes with different
> magnitudes; for /ja/, /ji/, /ju/ the activity was similar before the C-V transition, while a
> dorsal electrode was active during /u/ only.

The judged part: in at least one subject, a vSMC electrode with a positive lips weight (from the
GLM electrodes of claim fig2a-lips-dorsal) has mean high gamma above 2 at +250 ms for /ju/ and
below 2 for /ja/ and /ji/. The lips weights and the GLM electrodes are computed without electrode
positions, which are only needed for fig2a's spatial judgement (see the summary).

In [ ]:
t250 = B.ms_index(250)
fh = []
for key in W_READINGS:
    base, cs = key[0], key[1]
    hits = {}
    for s in B.SUBJECTS:
        cvs = CVS[s, cs]
        Hj = {v: HV[s, base, cs][cvs.index(("y", v)), :, t250] for v in "aiu"}
        w = WEIGHTS[key][s]
        cand = w["keep"] & (w["W"][:, 0] > 0)
        hit = cand & (Hj["u"] > 2) & (Hj["a"] < 2) & (Hj["i"] < 2)
        hits[s] = (int(cand.sum()), int(hit.sum()), float(Hj["u"][cand].max()) if cand.any() else np.nan)
    fh.append({"baseline": base, "CVs": cs, "timing reading": " / ".join(key[2]), "p-value": key[3],
               **{f"{s} lips+ / hits / max /ju/": f"{hits[s][0]} / {hits[s][1]} / {hits[s][2]:.2f}" for s in B.SUBJECTS},
               "holds": any(h[1] > 0 for h in hits.values())})
fh = pd.DataFrame(fh)
print(fh.to_string(index=False))
print("\nReadings where it holds:", int(fh.holds.sum()), "of", len(fh))
print("Largest /ju/ z at +250 ms among lips-positive GLM electrodes, over all readings and subjects:",
      max(float(v.split(" / ")[2]) for c in fh.columns if "max /ju/" in c for v in fh[c]))
print("Outcome over readings:", overall(["reproduced" if h else "not reproduced" for h in fh.holds]))
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
key0 = W_READINGS[0]
for ax, s in zip(axes, B.SUBJECTS):
    cvs = CVS[s, "all"]
    H = HV[s, "pre", "all"]
    w = WEIGHTS[key0][s]
    cand = np.where(w["keep"] & (w["W"][:, 0] > 0))[0]
    d = H[cvs.index(("y", "u")), cand, t250] - np.maximum(H[cvs.index(("y", "a")), cand, t250], H[cvs.index(("y", "i")), cand, t250])
    e = cand[np.argmax(d)]
    tr = H[[cvs.index(("y", v)) for v in "aiu"], e]
    for v, row in zip("aiu", tr):
        ax.plot(B.EPOCH_MS, row, label=f"/j{v}/")
    ax.set_title(f"{s} electrode {S[s]['vsmc'][e]}")
    print(f"{s}: plotted electrode {S[s]['vsmc'][e]} (largest /ju/ minus /ja/,/ji/ at +250 ms among lips+ GLM electrodes), z at +250 ms: "
          + ", ".join(f"/j{v}/ {row[t250]:.2f}" for v, row in zip("aiu", tr)) + f"; plotted range {tr.min():.2f} to {tr.max():.2f}; "
          f"mean |difference| between the three before 0 ms {np.abs(tr[:, B.EPOCH_MS < 0] - tr[:, B.EPOCH_MS < 0].mean(0)).mean():.2f}")
    ax.axvline(0, color="k", lw=0.5); ax.axhline(2, color="grey", lw=0.5, ls="--")
axes[0].legend(); axes[0].set_ylabel("z (pre baseline)")
plt.show()

**Result: not reproduced.** Under all 64 readings no GLM electrode with a positive lips weight
reaches z = 2 for /ju/ at +250 ms in any subject; the largest /ju/ value among them, over all
readings and subjects, is 1.19. The /ju/ responses of the plotted electrodes are larger than /ja/ and
/ji/ at +250 ms, but well under the paper's activity threshold. The other parts of the claim are
not judged (no measure in the paper).

## Spatial claims (blocked)

fig2a-lips-dorsal, fig2b-somatotopic-sequence, suppfig3b-spatial-clustering,
suppfig3d-cross-subject, suppfig3f-dv-sequence, suppfig5-intermediate-organization and
suppfig7-no-latency-difference all need each electrode's anterior-posterior distance from the
central sulcus and dorsal-ventral distance from the Sylvian fissure (Methods). The dandiset gives
neither. EC2 and GP31 have x, y, z coordinates with no unit, frame or description; GP33, one of
the three subjects, has none. The labels say precentral or postcentral, not how far from either
sulcus, and no electrode is labelled as on the guenon. These claims are blocked as
metadata_insufficient.

The GLM electrode selection and weights of fig2a-lips-dorsal do not need positions and are
computed above. For information only (not an outcome, and not a reading of the claim), the median
z coordinate of the top 10% of each articulator's weights in the two subjects that have
coordinates, under the first reading:

In [ ]:
key0 = W_READINGS[0]
print("reading:", key0)
for s in ("EC2", "GP31"):
    w = WEIGHTS[key0][s]
    z = S[s]["z"][S[s]["vsmc"]]
    kept = np.where(w["keep"])[0]
    out = []
    for a, name in enumerate(B.ARTICULATORS):
        if np.isnan(w["W"][kept, a]).all():
            out.append(f"{name} no window"); continue
        top = kept[np.argsort(-w["W"][kept, a])[:max(1, len(kept) // 10)]]
        out.append(f"{name} {np.median(z[top]):.1f}")
    print(f"{s}: {len(kept)} GLM electrodes; median z of top 10% weights: " + ", ".join(out))

## suppfig6-mixed-tuning

> Single electrodes show a clear preferred articulator and also carry weights for other
> articulators.

In [ ]:
s6 = []
for key in W_READINGS:
    Wk = np.concatenate([WEIGHTS[key][s]["W"][WEIGHTS[key][s]["keep"]] for s in B.SUBJECTS])
    no_window = int(np.isnan(Wk).any(1).sum())
    Wk = np.nan_to_num(Wk, nan=-np.inf)
    pos = np.where(Wk > 0, Wk, 0)
    has = pos.sum(1) > 0
    pct = 100 * pos[has] / pos[has].sum(1, keepdims=True)
    pref = np.argmax(Wk[has], axis=1)
    groups, ok = {}, True
    for a in range(4):
        g = pct[pref == a]
        if len(g) == 0:
            groups[B.ARTICULATORS[a]] = "none"
            continue
        m = g.mean(0)
        ok &= bool(np.argmax(m) == a and (np.delete(m, a) > 0).any())
        groups[B.ARTICULATORS[a]] = f"n={len(g)} " + "/".join(f"{x:.0f}" for x in m)
    s6.append({"baseline": key[0], "CVs": key[1], "timing reading": " / ".join(key[2]), "p-value": key[3],
               "electrodes": int(len(Wk)), "with an articulator lacking T_A": no_window,
               "no positive weight": int((~has).sum()), **groups, "holds": ok})
s6 = pd.DataFrame(s6)
print("Group means of positive weights as % (Lips/Tongue/Larynx/Jaw), electrodes grouped by largest W:")
print(s6.to_string(index=False))
print("\nReadings where it holds:", int(s6.holds.sum()), "of", len(s6))
print("Outcome over readings:", overall(["reproduced" if h else "not reproduced" for h in s6.holds]))

**Result: reproduced.** Under every reading each group's preferred articulator has the largest
mean share of positive weight (Lips 63 to 79%, Tongue 53 to 96%, Larynx 59 to 84%, Jaw 72 to
84%), and the other articulators keep a non-zero share (table). In readings where an articulator
has no window T_A in a subject (GP31's tongue, in the "all six" threshold readings), that
articulator's weight is missing for the subject's electrodes; the count is in the table.

## fig3d-timing-sequence

> Articulator correlations follow a staggered sequence consistent across subjects; lip and tongue
> correlations begin well before sound onset, jaw and larynx align to the C-V transition, and the
> high and back tongue vowel features peak near the vowel midpoint (~250 ms).

Acoustic onset: the dandiset has no audio, so the onset of each CV is taken from the hand-marked
syllable start of the trial table (mean over renditions, then over CVs). This replaces the
prototype and thresholding of the Methods (a deviation, asked as a question).

In [ ]:
onsets = []
for s in B.SUBJECTS:
    for b in blocks[s]:
        k = b["kept"]
        for lab, st, cv in zip(b["condition"][k], b["start"][k], b["cv"][k]):
            p = B.parse_label(lab)
            if p:
                onsets.append({"subject": s, "cv": p, "onset_ms": 1000 * (st - cv)})
onsets = pd.DataFrame(onsets)
per_cv = onsets.groupby(["subject", "cv"]).onset_ms.mean()
ACOUSTIC_ONSET = per_cv.mean()
print(f"Acoustic onset from the annotations: mean over {len(per_cv)} CVs {ACOUSTIC_ONSET:.1f} ms "
      f"(per subject {per_cv.groupby(level=0).mean().round(1).to_dict()}; CV range {per_cv.min():.0f} to {per_cv.max():.0f} ms)")
f3 = []
for base in BASES:
    for cs in CVSETS:
        for reading in TIMING_READINGS:
            t = TIMING[base, cs, reading]
            for anova in ("onset", "peak"):
                F, p = stats.f_oneway(*[t[t.articulator == a][anova] for a in ART6])
                mo = t.groupby("articulator").onset.mean()
                mp = t.groupby("articulator").peak.mean()
                c1 = mo["Lips"] < ACOUSTIC_ONSET and mo["Tongue"] < ACOUSTIC_ONSET
                c2 = min(mo["Jaw"], mo["Larynx"]) > max(mo["Lips"], mo["Tongue"])
                c3 = all(abs(mp[a] - 250) <= 25 for a in B.VOWEL_ARTICULATORS)
                n_ok = c1 + c2 + c3
                outcome = ("reproduced" if p < 1e-10 and n_ok == 3 else
                           "partially" if p < 1e-10 and n_ok == 2 else "not reproduced")
                f3.append({"baseline": base, "CVs": cs, "timing reading": " / ".join(reading), "ANOVA on": anova,
                           "n": len(t), **{f"n {a[:6]}": int((t.articulator == a).sum()) for a in ART6},
                           "F": round(F, 1), "p": p, **{f"onset {a[:6]}": round(mo.get(a, np.nan)) for a in ART6},
                           "peak Back": round(mp["Back Tongue"]), "peak High": round(mp["High Tongue"]),
                           "lips,tongue < acoustic": c1, "jaw,larynx later": c2, "vowel peaks 250+-25": c3,
                           "outcome": outcome})
f3 = pd.DataFrame(f3)
print(f3.to_string(index=False))
print("\nOutcome over readings:", overall(f3.outcome))
print("Ranges over readings: mean onset Lips", f3["onset Lips"].min(), "to", f3["onset Lips"].max(),
      "| Tongue", f3["onset Tongue"].min(), "to", f3["onset Tongue"].max(), "| Jaw", f3["onset Jaw"].min(), "to", f3["onset Jaw"].max(),
      "| Larynx", f3["onset Larynx"].min(), "to", f3["onset Larynx"].max(), "| peak Back", f3["peak Back"].min(), "to", f3["peak Back"].max(),
      "| peak High", f3["peak High"].min(), "to", f3["peak High"].max())
print("ANOVA F", f3.F.min(), "to", f3.F.max(), "| readings with p < 1e-10:", int((f3.p < 1e-10).sum()), "of", len(f3),
      "| conditions holding per reading:", (f3["lips,tongue < acoustic"].astype(int) + f3["jaw,larynx later"] + f3["vowel peaks 250+-25"]).value_counts().to_dict())
print("Total electrodes over articulators and subjects: from", f3.n.min(), "to", f3.n.max(), "(paper 211 +- 21)")
print("Per subject, first reading (pre, all CVs):", TIMING["pre", "all", TIMING_READINGS[0]].groupby(["subject", "articulator"]).size().unstack().to_dict("index"))
for s in B.SUBJECTS:
    t = TIMING["pre", "all", TIMING_READINGS[0]]
    print(s, "mean onsets (ms):", t[t.subject == s].groupby("articulator").onset.mean().round().to_dict())
fig, ax = plt.subplots(figsize=(6, 3))
t = TIMING["pre", "all", TIMING_READINGS[0]]
for i, a in enumerate(ART6):
    g = t[t.articulator == a]
    ax.errorbar(g.onset.mean(), i, xerr=g.onset.sem(), fmt="<", color="C0")
    ax.errorbar(g.peak.mean(), i, xerr=g.peak.sem(), fmt="o", color="C1")
    ax.errorbar(g.offset.mean(), i, xerr=g.offset.sem(), fmt=">", color="C2")
ax.axvline(ACOUSTIC_ONSET, color="grey", ls="--"); ax.axvline(0, color="k", lw=0.5)
ax.set_yticks(range(6)); ax.set_yticklabels(ART6); ax.set_xlabel("ms (first reading, pre baseline, all CVs)")
plt.show()

**Result: not reproduced.** Under every one of the 64 readings none of the three order conditions
holds. The lips correlations begin before the annotated acoustic onset (-110 ms; mean lips onset
-270 to -54 ms) but the tongue correlations begin at or after the C-V transition (-26 to +220 ms);
the jaw begins as early as the lips (-270 to -116 ms), not later; and the high-tongue peak is at
276 to 317 ms, outside 250 +- 25 ms. The ANOVA across the six articulators has F = 9.9 to 39.2,
with p below 1e-10 in 51 of the 64 readings. Electrode counts are 162 to 226 (paper 211).

## pca-nine-components

> The cortical state-space has nine spatial PCs in every subject, the eigenvalues falling fast up
> to the ninth and slowly after, together explaining about 60% of the variance.

In [ ]:
pc = []
for base in BASES:
    for z in ZMODES:
        for cs in CVSETS:
            for w in [(-500, 750), (-375, 620)]:
                fr = {s: STATE[s, base, z, cs, w][1] for s in B.SUBJECTS}
                pc.append({"baseline": base, "z-scoring": z, "CVs": cs, "window": f"{w[0]} to {w[1]}",
                           **{s: round(fr[s][:9].sum(), 3) for s in B.SUBJECTS},
                           "mean": round(np.mean([fr[s][:9].sum() for s in B.SUBJECTS]), 3)})
pc = pd.DataFrame(pc)
pc["within 0.54-0.66"] = pc["mean"].between(0.54, 0.66)
print(pc.to_string(index=False))
print("\nOutcome over readings:", overall(["reproduced" if x else "not reproduced" for x in pc["within 0.54-0.66"]]))
print("Rest-baseline variant (row, all CVs, epoch):", {s: round(float(B.state_space(HV[s, "rest", "all"], "row")[1][:9].sum()), 3) for s in B.SUBJECTS})
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, z in zip(axes, ZMODES):
    for s in B.SUBJECTS:
        ax.plot(np.arange(1, 21), 100 * STATE[s, "pre", z, "all", EP][1][:20], "o-", ms=3, label=s)
    ax.axvline(9, color="grey", ls="--"); ax.set_title(f"pre baseline, {z} z-scoring, all CVs"); ax.set_xlabel("PC"); ax.set_ylabel("% variance")
axes[0].legend(); plt.show()
print("% variance of PCs 1 to 12, pre baseline, all CVs, epoch (all values plotted are within 0 to 40%):")
for z in ZMODES:
    for s in B.SUBJECTS:
        print(z, s, np.round(100 * STATE[s, "pre", z, "all", EP][1][:12], 1))

**Result: blocked (methods_underspecified).** With each electrode's concatenated series z-scored
(`row`), the first nine PCs explain a mean of 0.60 to 0.64 over subjects under every baseline, CV
set and window, within 0.54 to 0.66. With each electrode z-scored across syllables at each time
(`time`), the mean is 0.49 to 0.55, below the tolerance in seven of eight readings. The two readings of the
z-scoring give different outcomes. The spectra show no sharp change at the ninth eigenvalue in
any subject (printed values); that part is not judged.

## fig4a-consonant-clusters

> At the consonant time (-25 ms) k-means clustering of the cortical state-space groups syllables
> by major oral articulator, and three clusters are more parsimonious than two or four (mean
> silhouette 0.47 against 0.45 and 0.43).

In [ ]:
tc, tv = B.ms_index(-25), B.ms_index(250)
sil = []
for base, z, cs in SS_READINGS:
    for s in B.SUBJECTS:
        K = STATE[s, base, z, cs, EP][2][:, :, tc]
        row = {"baseline": base, "z-scoring": z, "CVs": cs, "subject": s}
        for k in (2, 3, 4):
            row[f"k={k}"] = silhouette_score(K, B.kmeans(K, k, seed=40 + k))
        sil.append(row)
sil = pd.DataFrame(sil)
print(sil.round(3).to_string(index=False))
means = sil.groupby(["baseline", "z-scoring", "CVs"])[["k=2", "k=3", "k=4"]].mean()
means["k=3 highest"] = means.idxmax(axis=1) == "k=3"
means["all within 10%"] = (abs(means["k=2"] - 0.45) <= 0.045) & (abs(means["k=3"] - 0.47) <= 0.047) & (abs(means["k=4"] - 0.43) <= 0.043)
print("\nMean over subjects (paper 0.45, 0.47, 0.43):")
print(means.round(3).to_string())
print("Outcome over readings:", overall(["reproduced" if h and w else "partially" if h else "not reproduced"
                                         for h, w in zip(means["k=3 highest"], means["all within 10%"])]))

**Result: not reproduced.** Three clusters do not have the highest mean silhouette under any
reading, and the silhouettes are lower than the paper's (k = 3: 0.19 to 0.28 against 0.47). With
every CV, GP31's k = 2 silhouette is near 0.8 because its /thi/, with two renditions, stands
apart from all other CVs; with only CVs of 15 or more renditions this goes away, and k = 3 is
still not the most parsimonious.

## suppfig11-co-clustering

> Clusters of the cortical state-space match the major articulators at the consonant time
> (median co-clustering 0.50 against 0.32 by chance) and the vowels at the vowel time (0.65
> against 0.37), across subjects.

In [ ]:
rng = np.random.default_rng(11)
cc = []
for base, z, cs in SS_READINGS:
    obs = {"cons a": [], "cons b": [], "vowel": []}
    null = {"cons a": [], "cons b": [], "vowel": []}
    for s in B.SUBJECTS:
        K = STATE[s, base, z, cs, EP][2]
        nc_c = B.kmeans(K[:, :, tc], 3, seed=21)
        nc_v = B.kmeans(K[:, :, tv], 3, seed=22)
        for name, nc, labs in (("cons a", nc_c, [B.MAJOR["a"].get(c) for c, v in CVS[s, cs]]),
                               ("cons b", nc_c, [B.MAJOR["b"].get(c) for c, v in CVS[s, cs]]),
                               ("vowel", nc_v, [v for c, v in CVS[s, cs]])):
            keep = np.array([l is not None for l in labs])
            ac = np.array([l for l in labs if l is not None])
            ncs = nc[keep]
            obs[name].append(B.co_clustering(ncs, ac))
            null[name].append(np.mean([B.co_clustering(ncs, rng.permutation(ac)) for _ in range(1000)], axis=0))
    row = {"baseline": base, "z-scoring": z, "CVs": cs}
    ok_tests, ok_med = True, True
    for name in obs:
        o, n = np.concatenate(obs[name]), np.concatenate(null[name])
        p = stats.wilcoxon(o, n).pvalue
        row[f"{name}: n, obs, null, p"] = f"{len(o)}, {np.median(o):.3f}, {np.median(n):.3f}, {p:.1e}"
        ok_tests &= bool(p < 1e-10 and np.median(o - n) > 0)
    for cons in ("cons a", "cons b"):
        mo = {k: np.median(np.concatenate(obs[k])) for k in (cons, "vowel")}
        mn = {k: np.median(np.concatenate(null[k])) for k in (cons, "vowel")}
        within = abs(mo[cons] - 0.50) <= 0.05 and abs(mn[cons] - 0.32) <= 0.032 and abs(mo["vowel"] - 0.65) <= 0.065 and abs(mn["vowel"] - 0.37) <= 0.037
        row[f"outcome ({cons[-1]})"] = "reproduced" if ok_tests and within else ("partially" if ok_tests else "not reproduced")
    cc.append(row)
cc = pd.DataFrame(cc)
print(cc.to_string(index=False))
print("Expected: consonants 0.50 +- 0.05 against 0.32 +- 0.032; vowels 0.65 +- 0.065 against 0.37 +- 0.037; p < 1e-10")
print("Outcome over readings:", overall(list(cc["outcome (a)"]) + list(cc["outcome (b)"])))

**Result: blocked (methods_underspecified).** The state-space clusters match the major
articulators more than chance under every reading (p from 3e-20 to 6e-16 for consonants). The
observed consonant medians (0.60 to 0.78) are above the paper's 0.50 +- 0.05, and the vowel
medians (0.41 to 0.65) are below its 0.65 +- 0.065 in most readings, so 12 of the 16 readings
give partially reproduced. With only CVs of 15 or more renditions and `row` z-scoring, the vowel
test gives p = 5e-08 and 3e-08, which misses the p < 1e-10 bound, and those 4 readings give not
reproduced.

## fig4c-d-hierarchy

> Hierarchical clustering of the state-space puts the major oral articulator at the top tier for
> consonants; for vowels the first split is lip rounding (/u/ against /a/ and /i/).

In [ ]:
hc = []
for base, z, cs in SS_READINGS:
    for s in B.SUBJECTS:
        K = STATE[s, base, z, cs, EP][2]
        cons_cut = fcluster(linkage(K[:, :, tc], "ward"), 3, "maxclust")
        vow_cut = fcluster(linkage(K[:, :, tv], "ward"), 2, "maxclust")
        row = {"baseline": base, "z-scoring": z, "CVs": cs, "subject": s}
        for lr in ("a", "b"):
            labs = np.array([B.MAJOR[lr].get(c, "") for c, v in CVS[s, cs]])
            lab = labs != ""
            maj = {k: pd.Series(labs[lab & (cons_cut == k)]).mode().tolist() for k in (1, 2, 3)}
            distinct = all(len(v) == 1 for v in maj.values()) and len({v[0] for v in maj.values()}) == 3
            home = {a: pd.Series(cons_cut[lab & (labs == a)]).mode()[0] for a in ("labial", "coronal", "dorsal")}
            purity = np.mean([cons_cut[i] == home[labs[i]] for i in np.where(lab)[0]])
            row[f"cons ({lr}) majorities"] = "/".join(v[0] if len(v) == 1 else "tie" for v in maj.values())
            row[f"cons ({lr}) in home"] = round(purity, 2)
            row[f"cons ({lr}) holds"] = bool(distinct and purity >= 2 / 3)
            row[f"cons ({lr}) co-clust"] = round(np.median(B.co_clustering(cons_cut[lab], labs[lab])), 2)
        vow = np.array([v for c, v in CVS[s, cs]])
        ok = any((vow_cut[vow == "u"] == ku).mean() > 0.5 and (vow_cut[vow != "u"] == 3 - ku).mean() > 0.5 for ku in (1, 2))
        row["a/i/u in cluster 1"] = "/".join(f"{(vow_cut[vow == v] == 1).mean():.2f}" for v in "aiu")
        row["vowel holds"] = bool(ok)
        hc.append(row)
hc = pd.DataFrame(hc)
print(hc.to_string(index=False))
oc = []
for (base, z, cs), g in hc.groupby(["baseline", "z-scoring", "CVs"]):
    for lr in ("a", "b"):
        both = g[f"cons ({lr}) holds"] & g["vowel holds"]
        oc.append({"baseline": base, "z-scoring": z, "CVs": cs, "labels": lr, "subjects with both": int(both.sum()),
                   "outcome": "reproduced" if both.all() else ("partially" if both.any() else "not reproduced")})
oc = pd.DataFrame(oc)
print(oc.to_string(index=False))
print("Outcome over readings:", overall(oc.outcome))

**Result: blocked (methods_underspecified).** The rule (both the consonant and the vowel split hold
in a subject) holds only in EC2, and only with every CV and `row` z-scoring (partially
reproduced, 4 readings). Under the other 12 readings no subject meets both; in most rows of the
table two of the three consonant clusters have a coronal majority. The vowel split (/u/ against /a/ and /i/) holds in EC2 under
most readings, and in GP31 under none.

## fig4ef-feature-correlations

> Over early times (-375 to 120 ms) constriction location correlates with the state-space
> structure more than constriction degree, which correlates more than the upcoming vowel; over
> later times (125 to 620 ms) vowel configuration correlates most.

In [ ]:
FEATS = ("location", "degree", "vowel")


def feature_r(base, z, cs, metric="euclidean"):
    """R(t) per subject and feature set (Methods Eq. 7): {(subject, set): R over epoch times}."""
    out = {}
    for s in B.SUBJECTS:
        K = STATE[s, base, z, cs, EP][2]
        dK = np.array([pdist(K[:, :, t]) for t in range(K.shape[2])])
        for f in FEATS:
            dP = pdist(B.feature_matrix(CVS[s, cs], f), metric)
            out[s, f] = np.array([np.corrcoef(d, dP)[0, 1] for d in dK])
    return out


early = (B.EPOCH_MS >= -375) & (B.EPOCH_MS < 120)
late = (B.EPOCH_MS >= 125) & (B.EPOCH_MS < 620)
RT = {k: feature_r(*k) for k in SS_READINGS}
fe = []
for (base, z, cs), R in RT.items():
    for metric in ("euclidean", "hamming"):
        Rm = R if metric == "euclidean" else feature_r(base, z, cs, "hamming")
        e = {f: np.concatenate([Rm[s, f][early] for s in B.SUBJECTS]) for f in FEATS}
        l = {f: np.concatenate([Rm[s, f][late] for s in B.SUBJECTS]) for f in FEATS}
        tests = {"early loc>deg": (e["location"], e["degree"]), "early deg>vow": (e["degree"], e["vowel"]),
                 "late vow>loc": (l["vowel"], l["location"]), "late vow>deg": (l["vowel"], l["degree"])}
        row = {"baseline": base, "z-scoring": z, "CVs": cs, "distance": metric, "n": len(e["location"])}
        n_ok = 0
        for name, (a, b) in tests.items():
            p = stats.wilcoxon(a, b).pvalue
            ok = bool(np.median(a - b) > 0 and p < 1e-10)
            row[name] = f"{np.median(a - b):+.3f} p={p:.0e} {'Y' if ok else 'n'}"
            n_ok += ok
        row["med early L/D/V"] = "/".join(f"{np.median(e[f]):.2f}" for f in FEATS)
        row["med late L/D/V"] = "/".join(f"{np.median(l[f]):.2f}" for f in FEATS)
        row["outcome"] = "reproduced" if n_ok == 4 else ("partially" if n_ok else "not reproduced")
        fe.append(row)
fe = pd.DataFrame(fe)
print(fe.to_string(index=False))
print("Outcome over readings (Euclidean; Hamming is a variant):", overall(fe[fe.distance == "euclidean"].outcome))

**Result: blocked (methods_underspecified).** In the early window, constriction location
correlates with the state-space distances more than constriction degree, and degree more than the
vowel, with p < 1e-10 under every reading with Euclidean distances (n = 297, 99 times x 3
subjects). In the late window the result depends on the CV set: with every CV, location
correlates more than the vowel features (partially reproduced); with only CVs of 15 or more
renditions the vowel features correlate more than location in every reading, and more than
degree with p < 1e-10 under `time` z-scoring (reproduced) but not under `row` (partially). With
Hamming distances (a variant), the early location-over-degree test misses 1e-10 with 15+ CVs.

## fig5c-separability-timing

> Cluster separability, averaged over subjects, rises from about -300 ms and peaks about 200 ms
> before the C-V transition for consonants, and rises from about 50 ms and peaks at +250 ms for
> vowels.

In [ ]:
rng5 = np.random.default_rng(5)
N_BOOT = 1000


def cluster_labels(s, cs, K, which, kind):
    """Labels per CV (per time and CV for k-means); -1 for CVs without a label."""
    cvs = CVS[s, cs]
    if kind == "kmeans":
        return np.array([B.kmeans(K[:, :, t], 3, seed=1000 + t, n_init=10) for t in range(K.shape[2])])
    if which == "vowel":
        return np.array(["aiu".index(v) for c, v in cvs])
    lr = kind[-1]
    names = {"labial": 0, "coronal": 1, "dorsal": 2}
    out = []
    for c, v in cvs:
        m = B.MAJOR[lr].get(c)
        out.append(3 if kind.startswith("four") and c in B.SIBILANTS else (names[m] if m else -1))
    return np.array(out)


SEP_KINDS = {"consonant": ["groups a", "groups b", "four a", "four b", "kmeans"], "vowel": ["groups", "kmeans"]}


def separability_curves(base, z, cs):
    out = {}
    kmeans_cache = {}
    for which, kinds in SEP_KINDS.items():
        for kind in kinds:
            curves, boots = [], []
            for s in B.SUBJECTS:
                K = STATE[s, base, z, cs, EP][2]
                if kind == "kmeans":
                    if s not in kmeans_cache:
                        kmeans_cache[s] = cluster_labels(s, cs, K, which, kind)
                    lab = kmeans_cache[s]
                else:
                    lab = cluster_labels(s, cs, K, which, kind)
                keep = (lab >= 0) if lab.ndim == 1 else np.ones(K.shape[0], bool)
                Ks = K[keep]
                labk = lab[keep] if lab.ndim == 1 else lab[:, keep]
                D = np.array([squareform(pdist(Ks[:, :, t])) for t in range(Ks.shape[2])])
                curves.append(B.separability(D, labk)[0])
                wts = rng5.multinomial(len(Ks), np.ones(len(Ks)) / len(Ks), size=N_BOOT)
                boots.append(B.separability(D, labk, wts))
            out[which, kind] = (np.mean(curves, 0), np.mean(boots, 0).std(0), curves)
    return out


SEP = {k: separability_curves(*k) for k in SS_READINGS}


def onset_peak(curve, se):
    """Peak time, onset (a) first time above the level at -500 ms by 2 bootstrap s.e., onset (b)
    first time of the run up to the peak above half the peak value."""
    p = int(np.argmax(curve))
    above = np.where(curve[:p + 1] > curve[0] + 2 * se[:p + 1])[0]
    on_a = B.EPOCH_MS[above[0]] if len(above) else np.nan
    below = np.where(curve[:p + 1] < curve[p] / 2)[0]
    on_b = B.EPOCH_MS[below[-1] + 1] if len(below) else np.nan
    return B.EPOCH_MS[p], on_a, on_b


sp = []
for (base, z, cs), d in SEP.items():
    for ck in SEP_KINDS["consonant"]:
        vk = "kmeans" if ck == "kmeans" else "groups"
        cpk, con_a, con_b = onset_peak(d["consonant", ck][0], d["consonant", ck][1])
        vpk, von_a, von_b = onset_peak(d["vowel", vk][0], d["vowel", vk][1])
        peaks_ok = abs(cpk + 200) <= 20 and abs(vpk - 250) <= 25
        ons_ok = {"a": abs(con_a + 300) <= 30 and abs(von_a - 50) <= 10, "b": abs(con_b + 300) <= 30 and abs(von_b - 50) <= 10}
        sp.append({"baseline": base, "z-scoring": z, "CVs": cs, "clusters": f"{ck} | {vk}",
                   "cons peak": cpk, "cons onset a": con_a, "cons onset b": con_b,
                   "vowel peak": vpk, "vowel onset a": von_a, "vowel onset b": von_b,
                   "peaks ok": peaks_ok, "onset ok a": ons_ok["a"], "onset ok b": ons_ok["b"],
                   "outcome": "reproduced" if peaks_ok and any(ons_ok.values()) else ("partially" if peaks_ok else "not reproduced")})
sp = pd.DataFrame(sp)
print(sp.to_string(index=False))
print("Expected: consonant peak -200 +- 20, onset -300 +- 30; vowel peak 250 +- 25, onset 50 +- 10 ms")
print("Outcome over readings:", overall(sp.outcome))
g = sp[~sp.clusters.str.startswith("kmeans")]
print("Phonetic-group readings: consonant peak", g["cons peak"].min(), "to", g["cons peak"].max(), "| vowel peak", g["vowel peak"].min(), "to", g["vowel peak"].max(),
      "| consonant onset (a)", g["cons onset a"].min(), "to", g["cons onset a"].max(), "| (b)", g["cons onset b"].min(), "to", g["cons onset b"].max(),
      "| vowel onset (a)", g["vowel onset a"].min(), "to", g["vowel onset a"].max(), "| (b)", g["vowel onset b"].min(), "to", g["vowel onset b"].max())
g = sp[sp.clusters.str.startswith("kmeans")]
print("k-means readings: peak of the single curve", g["cons peak"].min(), "to", g["cons peak"].max())
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, key in zip(axes, [("pre", "row", "all"), ("pre", "time", "all")]):
    for (which, kind), (c, se, _) in SEP[key].items():
        if kind != "kmeans":
            ax.plot(B.EPOCH_MS, c, label=f"{which} {kind}")
    ax.axvline(0, color="k", lw=0.5); ax.set_title(", ".join(key)); ax.set_xlabel("ms")
axes[0].legend(fontsize=7); axes[0].set_ylabel("separability")
plt.show()
for key, d in SEP.items():
    print(key, "curve ranges:", {f"{w} {k}": (round(float(c.min()), 2), round(float(c.max()), 2)) for (w, k), (c, se, _) in d.items()})

**Result: not reproduced.** Under no reading are both peaks within tolerance. With the phonetic
groups the consonant separability peaks at -145 to +10 ms (paper -200 +- 20) and the vowel
separability at 200 to 235 ms (paper 250 +- 25, so within it in some readings); with k-means at
each time there is one curve for both, peaking anywhere from -135 to 360 ms. The consonant onsets
(-305 to -180 ms) reach the paper's -300 +- 30 ms under some readings, and the vowel onsets (60
to 105 ms) are later than its 50 +- 10 ms.

## fig5d-separability-coupling

> Cluster separability and the correlation of the state-space with phonetic features are tightly
> linked over time for consonants and vowels in each subject (R2 0.42 to 0.98, all P < 1e-10).

In [ ]:
c5 = []
for (base, z, cs), d in SEP.items():
    R = RT[base, z, cs]
    for ck in SEP_KINDS["consonant"]:
        vk = "kmeans" if ck == "kmeans" else "groups"
        for rmode in ("(a) matching set", "(b) mean of three"):
            vals = []
            for i, s in enumerate(B.SUBJECTS):
                for which, kind, fset in (("consonant", ck, "location"), ("vowel", vk, "vowel")):
                    r = R[s, fset] if rmode.startswith("(a)") else np.mean([R[s, f] for f in FEATS], 0)
                    res = stats.pearsonr(d[which, kind][2][i], r)
                    vals.append((res.statistic ** 2, res.pvalue, res.statistic))
            r2 = np.array([v[0] for v in vals]); ps = np.array([v[1] for v in vals]); rs = np.array([v[2] for v in vals])
            ok = bool((r2 >= 0.378).all() and (ps < 1e-10).all() and (rs > 0).all())
            part = bool((rs > 0).all() and (ps < 1e-10).all())
            c5.append({"baseline": base, "z-scoring": z, "CVs": cs, "clusters": f"{ck} | {vk}", "R": rmode,
                       "R2 (EC2 c,v GP31 c,v GP33 c,v)": " ".join(f"{x:.2f}" for x in r2),
                       "max p": ps.max(), "min r": round(rs.min(), 2),
                       "outcome": "reproduced" if ok else ("partially" if part else "not reproduced")})
c5 = pd.DataFrame(c5)
print(c5.to_string(index=False))
print("\nOutcome over readings:", overall(c5.outcome))
print("Outcomes by reading of the clusters and of R:")
print(c5.groupby(["clusters", "R"]).outcome.value_counts().unstack(fill_value=0).to_string())

**Result: blocked (methods_underspecified).** With the phonetic groups as clusters and the
matching feature set for R (constriction location for consonants, vowel features for vowels),
all six R2 values lie within 0.38 to 1 with p < 1e-10 in every baseline, z-scoring and CV
reading. With the mean of the three feature sets, or with k-means clusters at each time, the
coupling fails (R2 near 0 in some subjects, negative r). The readings of the claim give different
outcomes.

## suppfig13-consonant-vowel-distance

> Consonant and vowel representations occupy distinct regions of the state-space;
> consonant-vowel distances are larger than consonant-consonant and vowel-vowel distances, and
> consonant-consonant distances are larger than vowel-vowel ones.

In [ ]:
cvd = []
for base, z, cs in SS_READINGS:
    for reading in ("(a) mean of cross pairings", "(b) i at -25, j at +250"):
        CC, VV, CVd = [], [], []
        for s in B.SUBJECTS:
            K = STATE[s, base, z, cs, EP][2]
            Kc, Kv = K[:, :, tc], K[:, :, tv]
            i, j = np.triu_indices(len(Kc), 1)
            CC.append(np.linalg.norm(Kc[i] - Kc[j], axis=1))
            VV.append(np.linalg.norm(Kv[i] - Kv[j], axis=1))
            d1 = np.linalg.norm(Kc[i] - Kv[j], axis=1)
            d2 = np.linalg.norm(Kc[j] - Kv[i], axis=1)
            CVd.append((d1 + d2) / 2 if reading.startswith("(a)") else d1)
        CC, VV, CVd = map(np.concatenate, (CC, VV, CVd))
        row = {"baseline": base, "z-scoring": z, "CVs": cs, "CV distance": reading, "pairs": len(CC),
               "median CC": round(np.median(CC), 2), "median VV": round(np.median(VV), 2), "median CV": round(np.median(CVd), 2)}
        ok = True
        for name, (a, b) in {"CV>CC": (CVd, CC), "CV>VV": (CVd, VV), "CC>VV": (CC, VV)}.items():
            p = min(1.0, 3 * stats.wilcoxon(a, b).pvalue)
            row[name] = f"{np.median(a - b):+.2f} p={p:.0e}"
            ok &= bool(np.median(a - b) > 0 and p < 1e-10)
        row["orderings hold"] = ok
        row["outcome"] = ("reproduced" if ok and len(CC) == 4623 else "partially" if ok else "not reproduced")
        cvd.append(row)
cvd = pd.DataFrame(cvd)
print(cvd.to_string(index=False))
print("Pairs within subjects (all CVs):", {s: len(CVS[s, 'all']) * (len(CVS[s, 'all']) - 1) // 2 for s in B.SUBJECTS},
      "| 15+ CVs:", {s: len(CVS[s, '15+']) * (len(CVS[s, '15+']) - 1) // 2 for s in B.SUBJECTS}, "(paper n = 4623)")
print("Outcome over readings:", overall(cvd.outcome))

**Result: blocked (methods_underspecified).** With `row` z-scoring, consonant-vowel distances
exceed consonant-consonant ones, which exceed vowel-vowel ones, with corrected p < 1e-10 under
every other reading; with `time` z-scoring the consonant-vowel distances are shorter than the
consonant-consonant ones. The number of pairs is 4677 with every CV and 4560 with CVs of 15 or
more renditions, never the paper's 4623, so the `row` readings would give a partial outcome and
the `time` readings not reproduced.

## suppfig9-pc-articulator-weights

> The first three PCs are weighted toward tongue over lips, and toward lips over jaw and larynx,
> while jaw and larynx do not differ.

Note on the method: by Methods Eq. 6, log10|PCw_ij| = log10|M_ij| + log10|W_jA|, so for a given
electrode the PC term is the same for every articulator and cancels in the paired comparisons;
the tests give the same result in every PC.

In [ ]:
pairs6 = list(itertools.combinations(range(4), 2))
s9 = []
for (base, cs, reading, pmode) in W_READINGS:
    for z in ZMODES:
        for eset in ("all vSMC", "GLM electrodes"):
            if eset == "all vSMC" and pmode == "t-test":
                continue    # the p-value reading only changes which electrodes are kept
            vals = []
            for s in B.SUBJECTS:
                w = WEIGHTS[base, cs, reading, pmode][s]
                M = STATE[s, base, z, cs, EP][0]
                sel = w["keep"] if eset == "GLM electrodes" else np.ones(len(w["keep"]), bool)
                vals.append(np.log10(np.abs(M[sel, :3, None] * w["W"][sel, None, :])))
            V = np.concatenate(vals)            # (electrode, PC, articulator)
            V = V[np.isfinite(V).all(axis=(1, 2))]
            per_pc, meds = [], []
            for pci in range(3):
                p = {(a, b): (np.median(V[:, pci, a] - V[:, pci, b]), min(1.0, 6 * stats.wilcoxon(V[:, pci, a], V[:, pci, b]).pvalue))
                     for a, b in pairs6}

                def greater(a, b, alpha):
                    m, pv = p[(a, b)] if (a, b) in p else (-p[(b, a)][0], p[(b, a)][1])
                    return bool(m > 0 and pv < alpha)
                L, T, X, J = 0, 1, 2, 3
                ra = greater(L, J, 0.001) and greater(L, X, 0.001)
                rb = ra and greater(T, J, 0.001) and greater(T, X, 0.001)
                per_pc.append(greater(T, L, 0.05) and (ra or rb) and p[(X, J)][1] > 0.05)
                meds.append("/".join(f"{np.median(V[:, pci, a]):.2f}" for a in range(4)))
            n_ok = sum(per_pc)
            s9.append({"baseline": base, "CVs": cs, "timing reading": " / ".join(reading), "p-value": pmode, "z-scoring": z,
                       "electrodes": eset, "n": len(V), "PC1 medians L/T/X/J": meds[0], "PCs holding": "".join("Y" if x else "n" for x in per_pc),
                       "outcome": "reproduced" if n_ok == 3 else ("partially" if n_ok else "not reproduced")})
s9 = pd.DataFrame(s9)
print(s9.to_string(index=False))
print("\nOutcome over readings:", overall(s9.outcome))
base, cs, reading, pmode, z = "pre", "all", TIMING_READINGS[2], "F-test", "row"
V = np.concatenate([np.log10(np.abs(STATE[s, base, z, cs, EP][0][:, :3, None] * WEIGHTS[base, cs, reading, pmode][s]["W"][:, None, :])) for s in B.SUBJECTS])
V = V[np.isfinite(V).all(axis=(1, 2))]
for pci in range(3):
    ps = {f"{B.ARTICULATORS[a][:2]}-{B.ARTICULATORS[b][:2]}": f"{np.median(V[:, pci, a] - V[:, pci, b]):+.2f} p={min(1, 6 * stats.wilcoxon(V[:, pci, a], V[:, pci, b]).pvalue):.0e}" for a, b in pairs6}
    print(f"PC{pci + 1} ({base}, {cs}, {reading}, all vSMC, n={len(V)}): median paired differences and corrected p {ps}")

**Result: not reproduced.** Under every reading the condition fails in all three PCs. In the
reading printed in full (all 268 vSMC electrodes), the tongue weights are not larger than the
lips weights (median paired difference of log10 weights, lips minus tongue, +0.03, corrected
p = 1), and jaw and larynx differ (corrected p = 5e-11).

## Summary

| Claim | Outcome |
|---|---|
| data-composition | partially reproduced |
| active-electrodes | not reproduced |
| fig1c-e-articulator-electrodes | blocked: metadata_insufficient |
| fig1f-h-phonetic-contrasts | not reproduced |
| fig2a-lips-dorsal | blocked: metadata_insufficient |
| fig2b-somatotopic-sequence | blocked: metadata_insufficient |
| suppfig3b-spatial-clustering | blocked: metadata_insufficient |
| suppfig3d-cross-subject | blocked: metadata_insufficient |
| suppfig3f-dv-sequence | blocked: metadata_insufficient |
| suppfig5-intermediate-organization | blocked: metadata_insufficient |
| suppfig6-mixed-tuning | reproduced |
| fig3d-timing-sequence | not reproduced |
| suppfig7-no-latency-difference | blocked: metadata_insufficient |
| pca-nine-components | blocked: methods_underspecified |
| fig4a-consonant-clusters | not reproduced |
| suppfig11-co-clustering | blocked: methods_underspecified |
| fig4c-d-hierarchy | blocked: methods_underspecified |
| fig4ef-feature-correlations | blocked: methods_underspecified |
| fig5c-separability-timing | not reproduced |
| fig5d-separability-coupling | blocked: methods_underspecified |
| suppfig13-consonant-vowel-distance | blocked: methods_underspecified |
| suppfig9-pc-articulator-weights | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. High gamma is computed here from the raw ECoG, as
the Methods describe, with a filter bank whose widths the paper does not give (a standard choice)
and with the dataset's bad-channel flags in place of the paper's inspection. The baseline for the
z-scores is not given by the paper; both readings of the claims file are run. vSMC electrodes are
those labelled precentral or postcentral (268, against the paper's 266). The acoustic onset of
fig3d is the hand-marked syllable start, as the audio is not in the dandiset. Two method details
the claims file does not fix are run under both readings and asked as questions: how the
state-space input is z-scored, and whether CVs with fewer than 15 renditions are kept. Most of the
blocked outcomes for the state-space claims come from these readings giving different results,
and an answer could settle them. The GLM weights used by fig1f-h, suppfig6 and suppfig9 come from
claim fig2a-lips-dorsal, which is blocked for its spatial judgement only; the weights need no
positions, so these claims are tested.

What was found about the dataset. It holds four subjects; EC2, GP31 and GP33 fit the paper's
EC2, EC31 and EC33 (assumed, not shown), and EC9 is not in the paper. The audio was removed. No
electrode has a position relative to the central sulcus or the Sylvian fissure, and one of the
three subjects (GP33) has no coordinates at all, so seven spatial findings cannot be tested. The
coordinates of EC2 and GP31 have no unit or frame. GP31 has two renditions of /thi/ and none of
/tha/ or /thu/, and up to 107 renditions of other CVs; EC2 has 10 of /thi/; the paper states 15 to
100 per CV. Two labels are misspelt (`she`, `ha`). Electrode ids start at 0 in EC2 and GP31 and at 1
in GP33 and EC9. 177 kept trials overlap intervals marked as ECoG artefacts.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")